판정 임계값(yoy_up_ratio 0.75, 금액 하한 25% 분위)은 제가 임의로 정한 값이니 데이터를 보고 조정하세요.

In [11]:
import numpy as np
import pandas as pd
from scipy import stats

KEY = ["약효분류코드", "약효분류명"]


def _hac_slope_se(X, E, lags):
    """시계열별(E의 각 열) 추세 계수(X의 2번째 열)의 Newey-West(HAC) 표준오차."""
    T, k = X.shape
    XtXi = np.linalg.inv(X.T @ X)
    se = np.empty(E.shape[1])
    for i in range(E.shape[1]):
        u = X * E[:, [i]]
        S = u.T @ u
        for l in range(1, lags + 1):
            G = u[l:].T @ u[:-l]
            S += (1 - l / (lags + 1)) * (G + G.T)
        V = XtXi @ S @ XtXi * T / (T - k)   # 소표본 보정
        se[i] = np.sqrt(V[1, 1])
    return se


def growth_metrics(df, val="사용금액_백만원", window=36, end=None, min_y3=None, hac_lags=6):
    """36개월 창 성장지표.
    end='2025-12' 처럼 창의 마지막 월을 지정(미지정 시 데이터 마지막 월).
    - valid=False: 창 내 결측/0이 있는 분류군(신규·소멸 등) → 지표 계산 제외, grade='X'
    - grade: A=지속성장+가속, B=지속성장(가속 아님), C=후보이나 지속성 미확인, '-'=해당없음, X=제외
    """
    p = df.pivot_table(index=KEY, columns="진료년월", values=val, aggfunc="sum").sort_index(axis=1)
    if end:
        p = p.loc[:, :end]
    p = p.iloc[:, -window:]
    assert p.shape[1] == window, "월 수 부족"

    # 시장 총량은 전체 분류군 기준 (결측 = 해당 월 판매 없음으로 합산)
    mm = p.fillna(0)
    mk1, mk2, mk3 = (mm.iloc[:, i * 12:(i + 1) * 12].sum().sum() for i in range(3))

    # [수정3] 36개월 전부 존재하고 양수인 분류군만 분석 (fillna(0)로 무한대/-100% 성장 생성 방지)
    valid = p.notna().all(axis=1) & (p > 0).all(axis=1)
    m = p[valid]

    # 1) 연도별 성장률, 2) CAGR
    y1, y2, y3 = (m.iloc[:, i * 12:(i + 1) * 12].sum(axis=1) for i in range(3))
    out = pd.DataFrame({"Y1": y1, "Y2": y2, "Y3": y3})
    out["g_Y2Y1"] = y2 / y1 - 1
    out["g_Y3Y2"] = y3 / y2 - 1
    out["CAGR_2y"] = (y3 / y1) ** 0.5 - 1

    # 3) 시장 초과성장률, 증가 기여도
    out["excess_g_Y3Y2"] = out["g_Y3Y2"] - (mk3 / mk2 - 1)
    out["excess_CAGR"] = out["CAGR_2y"] - ((mk3 / mk1) ** 0.5 - 1)
    out["contrib_Y3Y2"] = (y3 - y2) / (mk3 - mk2)

    # 4) 월별 로그 추세 기울기 (계절 더미 OLS) + [수정2] HAC 표준오차 기반 95% CI
    t = np.arange(window)
    mon = np.array([int(c[-2:]) for c in m.columns])
    dummies = (mon[:, None] == np.arange(2, 13)[None, :]).astype(float)  # 1월 기준
    X = np.column_stack([np.ones(window), t, dummies])
    L = np.log(m.T.values)                                               # (window, n), 전부 양수
    beta = np.linalg.pinv(X) @ L
    se = _hac_slope_se(X, L - X @ beta, hac_lags)
    tc = stats.t.ppf(0.975, window - X.shape[1])
    out["trend_mom"] = np.expm1(beta[1])
    out["trend_lo"] = np.expm1(beta[1] - tc * se)
    out["trend_hi"] = np.expm1(beta[1] + tc * se)
    # 이상치 걱정 시: stats.theilslopes(np.log(row), t)[0] 로 대체

    # 5) 최근 12개월 중 전년 동월 대비 증가 개월 비율
    out["yoy_up_ratio"] = (m.iloc[:, -12:].values > m.iloc[:, -24:-12].values).mean(axis=1)

    # 6) 최근 TTM 성장률과 방향 (보조용)
    ttm = m.T.rolling(12).sum().T
    ttm_g = (ttm / ttm.shift(12, axis=1) - 1).iloc[:, -13:]
    out["ttm_g_last"] = ttm_g.iloc[:, -1]
    out["ttm_g_change"] = ttm_g.iloc[:, -1] - ttm_g.iloc[:, 0]

    # [수정1] 가속을 필수 조건에서 분리: 지속성장(후보) → 지속성 확인 → 가속 여부로 등급
    big = out["Y3"] >= (min_y3 if min_y3 is not None else out["Y3"].quantile(0.25))
    out["candidate"] = (
        big & (out["g_Y2Y1"] > 0) & (out["g_Y3Y2"] > 0)
        & (out["excess_g_Y3Y2"] > 0) & (out["contrib_Y3Y2"] > 0)
    )
    out["accel"] = out["g_Y3Y2"] > out["g_Y2Y1"]
    out["persistent"] = out["candidate"] & (out["trend_lo"] > 0) & (out["yoy_up_ratio"] >= 0.75)
    out["grade"] = np.select(
        [out.persistent & out.accel, out.persistent, out.candidate], ["A", "B", "C"], "-"
    )

    # 제외된 분류군도 행으로 남겨 표시
    out = out.reindex(p.index)
    out["valid"] = valid
    for c in ["candidate", "accel", "persistent"]:
        out[c] = out[c].fillna(False).astype(bool)
    out["grade"] = out["grade"].fillna("X")
    return out.reset_index()


# --- 사용 예 ---
# df = pd.read_csv("data.csv")
# amt = growth_metrics(df, "사용금액_백만원")
# qty = growth_metrics(df, "수량_천개")
# both = amt.merge(qty, on=KEY, suffixes=("_amt", "_qty"))
# both = both[both.grade_amt.isin(["A", "B"]) & both.grade_qty.isin(["A", "B"])]
# 확인용: growth_metrics(df, end="과거 시점 월")  # 과거 창에서 선정 → 이후 실제 성장과 비교

In [12]:
df = pd.read_csv("../meft_2014-2025.csv")
df['약효분류코드'] = df['약효분류코드'].astype(int).astype(str)

# 2025.12 기준 최근 36개월 목록 생성 (2023년 1월 ~ 2025년 12월)
months_36 = [f'{y}-{m:02d}' for y in range(2023, 2026) for m in range(1, 13)]
df_36 = df[df['진료년월'].isin(months_36)]

# 약효분류코드별, 진료년월별 피벗 테이블 생성
piv_amt = df_36.pivot(
    index='약효분류코드', columns='진료년월', values='사용금액_백만원'
)
piv_qty = df_36.pivot(
    index='약효분류코드', columns='진료년월', values='수량_천개'
)

all_codes = df['약효분류코드'].unique()
piv_amt = piv_amt.reindex(index=all_codes, columns=months_36)
piv_qty = piv_qty.reindex(index=all_codes, columns=months_36)

# 사용금액 또는 수량이 결측치(Null)인 경우를 True로 하는 마스크 생성
null_mask = piv_amt.isna() | piv_qty.isna()

# 약효분류코드와 약효분류명 매핑 딕셔너리
code_name_map = df.drop_duplicates('약효분류코드').set_index('약효분류코드')[
    '약효분류명'
]

# 1. 최근 36개월 동안 사용금액, 사용량 값이 null이 없는 분류코드
no_null_36 = null_mask.sum(axis=1) == 0
codes_no_null = piv_amt.index[no_null_36].tolist()

In [13]:
pool_df = df[df['약효분류코드'].isin(codes_no_null+['633', '250'])].copy()
amt = growth_metrics(pool_df, "사용금액_백만원")
qty = growth_metrics(pool_df, "수량_천개")
both = amt.merge(qty, on=KEY, suffixes=("_amt", "_qty"))

# both = both[both.persistent_amt & both.persistent_qty]   # 금액·수량 모두 통과 -> 없음

display(amt.head())
# display(qty.head())
# display(both.head())

,약효분류코드,약효분류명,Y1,Y2,Y3,g_Y2Y1,g_Y3Y2,CAGR_2y,excess_g_Y3Y2,excess_CAGR,...,trend_lo,trend_hi,yoy_up_ratio,ttm_g_last,ttm_g_change,candidate,accel,persistent,grade,valid
0,110,중추신경계용약,1868.0,3214.0,4122.0,0.720557,0.282514,0.485476,0.212648,0.422355,...,0.024866,0.045902,1.000000,0.282514,-0.438043,False,False,False,-,True
1,111,전신마취제,51584.0,47370.0,49933.0,-0.081692,0.054106,-0.016133,-0.015760,-0.079253,...,-0.004561,0.001873,0.833333,0.054106,0.135798,False,True,False,-,True
2,112,최면진정제,43657.0,45254.0,46723.0,0.036581,0.032461,0.034519,-0.037405,-0.028601,...,0.002107,0.003520,0.916667,0.032461,-0.004119,False,False,False,-,True
3,113,항전간제,274450.0,280733.0,288982.0,0.022893,0.029384,0.026133,-0.040482,-0.036987,...,0.001364,0.002868,0.750000,0.029384,0.006491,False,True,False,-,True
4,114,해열·진통·소염제,1037153.0,1053283.0,1072102.0,0.015552,0.017867,0.016709,-0.051999,-0.046411,...,0.000544,0.002250,0.583333,0.017867,0.002315,False,True,False,-,True
